In [1]:
#data pre-processing
import pandas as pd 
import numpy as np 
#data visualisation
import matplotlib.pyplot as plt 
import seaborn as sns
#to show max columns
pd.set_option('display.max_columns', None)
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
#for content base filtering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
#pip install scikit-surprise  #for recommedation system   but not using here because it wants python>3.10
#from surprise import Dataset, Reader

In [2]:
#Loading data
df_movies=pd.read_csv("../datasets/movies.dat",sep='::',encoding='latin-1',engine='python',names=['movieId', 'title', 'genres'])

df_ratings = pd.read_csv("../datasets/ratings.dat", sep="::", engine='python',names=['userId', 'movieId', 'rating', 'timestamp'])

df_users=pd.read_csv("../datasets/users.dat",sep="::",engine='python',names=['userId','gender','age','occupation','zip'])

In [3]:
#merging data
df=pd.merge(df_movies,df_ratings,on='movieId')
df=pd.merge(df,df_users,on='userId')

In [4]:
#Data Cleaning:

#changing data type of df_ratings dataset timestamp columns (int) to (date)
df['timestamp']=pd.to_datetime(df['timestamp'], unit='s')

In [5]:
#Some movie title start with ' or ... so we remove it 
df[df['title'].str.startswith((".", "'"," "), na=False)].head()

,movieId,title,genres,userId,rating,timestamp,gender,age,occupation,zip
194719,779,'Til There Was You (1997),Drama|Romance,38,2,2000-12-28 23:14:10,F,18,4,02215
194720,779,'Til There Was You (1997),Drama|Romance,45,1,2000-12-28 07:33:46,F,45,16,94110
194721,779,'Til There Was You (1997),Drama|Romance,156,4,2000-12-19 18:27:42,F,45,7,14519
194722,779,'Til There Was You (1997),Drama|Romance,338,3,2001-04-25 02:58:09,M,35,7,55116
194723,779,'Til There Was You (1997),Drama|Romance,517,4,2000-12-07 15:15:42,F,25,14,55408


In [6]:
#removing it 
df['title'] = df['title'].str.replace(r"^[\.\'\s ]+", "", regex=True).str.strip()
#so replacing "Children's -> Childrens ,Film-Noir -> FilmNoir ,Sci-Fi -> SciFI"
df['genres']=df['genres'].str.replace('|',' ')
#in tfidf any punctuation come then new word form from that single word
df['genres'] = df['genres'].str.replace("Children's", "Childrens")  #Because TF-IDF tokenization splits text into multiple tokens, especially when punctuation like hyphens or apostrophes are present
df['genres'] = df['genres'].str.replace("Sci-Fi", "SciFi")
df['genres'] = df['genres'].str.replace("Film-Noir", "FilmNoir")

In [20]:
#For streamlit app there is need of movie list so save inside joblib file
import os
os.makedirs("../models", exist_ok=True)
joblib.dump(df['title'].unique().tolist(), "../models/title.pkl")


['../models/title.pkl']

In [7]:
#'Weight calculating'
#counting total rating on each movies
counting=df.groupby('title')['rating'].count().reset_index().rename(columns={'rating':'count'})
averaging=df.groupby('title')['rating'].mean().reset_index().rename(columns={'rating':'mean'})
# 
df=pd.merge(df,counting,on='title')
df=pd.merge(df,averaging,on='title')

#ignoring movie who has total rating count is less than 50 
df=df[df['count']>50]
#Mean vote across the entire dataset
C=df['mean'].mean()
M=50  #select randomly

#create new feature for calculating Weightage of movie on rating 
df['Weight']=((df['mean']*df['count'])+C*M)/(df['count']+M)


In [8]:
df.head()

,movieId,title,genres,userId,rating,timestamp,gender,age,occupation,zip,count,mean,Weight
0,1,Toy Story (1995),Animation Childrens Comedy,1,5,2001-01-06 23:37:48,F,1,10,48067,2077,4.146846,4.133838
1,1,Toy Story (1995),Animation Childrens Comedy,6,4,2000-12-31 04:30:08,F,50,9,55117,2077,4.146846,4.133838
2,1,Toy Story (1995),Animation Childrens Comedy,8,4,2000-12-31 03:31:36,M,25,12,11413,2077,4.146846,4.133838
3,1,Toy Story (1995),Animation Childrens Comedy,9,5,2000-12-31 01:25:52,M,25,17,61614,2077,4.146846,4.133838
4,1,Toy Story (1995),Animation Childrens Comedy,10,5,2000-12-31 01:34:34,F,35,1,95370,2077,4.146846,4.133838


In [9]:
#collaborating model 

pivot_table_rating=pd.pivot_table(index='title',columns='userId',values='rating',data=df).fillna(0)
pivot_table_rating.head(2)

userId                             1     2     3     4     5     6     7     \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   0.0   0.0   0.0   

userId                             8     9     10    11    12    13    14    \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   0.0   0.0   0.0   0.0   1.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   0.0   0.0   0.0   

userId                             15    16    17    18    19    20    21    \
title                                                                         
10 Things I Hate About You (1999)   0.0   4.0   0.0   0.0   0.0   0.0   0.0   
101 Dalmatians (1961)               0.0   0.0   0.0   4.0   4.0   0.0   0.0   

userId                             22    23    24    25    26    27    28    \
title                                                                         
10 Things I Hate About You (1999)   2.0   0.0   0.0   0.0   0.0   0.0   0.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   0.0   0.0   0.0   

userId                             29    30    31    32    33    34    35    \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   0.0   0.0   0.0   4.0   0.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   0.0   0.0   0.0   

userId                             36    37    38    39    40    41    42    \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   4.0   4.0   0.0   0.0   0.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   0.0   0.0   0.0   

userId                             43    44    45    46    47    48    49    \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   0.0   0.0   0.0   

userId                             50    51    52    53    54    55    56    \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   0.0   3.0   0.0   0.0   0.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   0.0   0.0   0.0   

userId                             57    58    59    60    61    62    63    \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   0.0   0.0   0.0   0.0   2.0   
101 Dalmatians (1961)               0.0   3.0   0.0   0.0   0.0   0.0   0.0   

userId                             64    65    66    67    68    69    70    \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   0.0   0.0   0.0   0.0   4.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   0.0   0.0   0.0   

userId                             71    72    73    74    75    76    77    \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   4.0   0.0   4.0   0.0   3.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   2.0   0.0   0.0   

userId                             78    79    80    81    82    83    84    \
title                                                                         
10 Things I Hate About You (1999)   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
101 Dalmatians (1961)               0.0   0.0   0.0   0.0   0.0   0.0   0.0   

userId                             85    86    87    88    89    90    91    \
title                                                                         
10 Things I Hate About You (1999)   0.

In [10]:
#again use nearestneighbors model for collaborative recommedation 
from sklearn.neighbors import NearestNeighbors

collab_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=20,
    n_jobs=-1
)

collab_model.fit(pivot_table_rating)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",20
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1


In [11]:
#to remove repeated movie in title 
#“To ensure each movie is represented once, since content-based filtering operates on item features rather than user interactions.”
movie_df = df[['movieId', 'title', 'genres']].drop_duplicates()    #because Content-based recommendation works on items (movies)
movie_df.reset_index(drop=True, inplace=True)
movie_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation Childrens Comedy
1,2,Jumanji (1995),Adventure Childrens Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama
4,5,Father of the Bride Part II (1995),Comedy


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer  #(Term Frequency – Inverse Document Frequency)  common words → low importance,rare words → high importance
# Create TF-IDF object
tfidf = TfidfVectorizer(stop_words='english') 
# Fit and transform genres
tfidf_matrix = tfidf.fit_transform(movie_df['genres'])    #Machine Learning models cannot understand text  text → numeric
print(tfidf_matrix.shape)

#import cosine symil1rity
from sklearn.metrics.pairwise import cosine_similarity 
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)   #calculating similarity between movies.

print(cosine_sim.shape)

(2499, 18)
(2499, 2499)


### Top 100 movies by weight recommedation

In [13]:
df.head(2)

,movieId,title,genres,userId,rating,timestamp,gender,age,occupation,zip,count,mean,Weight
0,1,Toy Story (1995),Animation Childrens Comedy,1,5,2001-01-06 23:37:48,F,1,10,48067,2077,4.146846,4.133838
1,1,Toy Story (1995),Animation Childrens Comedy,6,4,2000-12-31 04:30:08,F,50,9,55117,2077,4.146846,4.133838


In [28]:
def get_top_n_rated_movies(n):
    return df[['title','Weight']].drop_duplicates().sort_values('Weight',ascending=False).head(n).reset_index(drop=True)

In [29]:
get_top_n_rated_movies(2)

,title,Weight
0,"Shawshank Redemption, The (1994)",4.533454
1,"Godfather, The (1972)",4.504476


In [38]:
def final_hybrid_recommendation(movie_name, no_of_recommendation=5):
    
    if movie_name not in pivot_table_rating.index:
        print(f"Your watched movie '{movie_name}' has total rating count either less than 50 or not in training data so not able to recommedate movie")
    else:
        print("Your selected movie :", movie_name)
        print("-"*60)
        
        # ----- Step 1: Collaborative Filtering -----
        movie_index = pivot_table_rating.index.get_loc(movie_name)
        distances, indices = collab_model.kneighbors(
            pivot_table_rating.iloc[movie_index,:].values.reshape(1,-1),
            n_neighbors=20
        )
        movie_indices = indices[0][1:]  # ignore selected movie
        hybrid_df = pd.DataFrame({"title": pivot_table_rating.index[movie_indices]})
        
        # ----- Step 2: Merge TF-IDF content similarity score -----
        # Compute cosine similarity of selected movie with candidates

        # Create mapping from movie title to index in TF-IDF matrix
        indices_content = pd.Series(movie_df.index, index=movie_df['title']).to_dict()
        
        idx = indices_content.get(movie_name, None)
        if idx is not None:
            sim_scores = list(enumerate(cosine_sim[idx]))                    #output[(0, np.float64(1.0000000000000002)),...]
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            sim_scores_dict = {movie_df['title'].iloc[i[0]]: i[1] for i in sim_scores}
            hybrid_df['content_sim'] = hybrid_df['title'].map(sim_scores_dict)    #map>>"For each title in hybrid_df['title'], find its value in sim_scores_dict and put it into content_sim column"
        else:
            hybrid_df['content_sim'] = 0
        
        # ----- Step 3: Merge popularity (Weighted Rating) -----
        hybrid_df = hybrid_df.merge(
            df[['title','Weight']].drop_duplicates(),
            on='title',
            how='left'
        )
        
        # ----- Step 4: Compute final score -----
        # Weighted sum: 50% collaborative, 30% content similarity, 20% popularity
        hybrid_df['final_score'] = (
            0.5 * (1 / (1 + distances[0][1:])) +  # collaborative (convert distance → similarity)
            0.3 * hybrid_df['content_sim'] +      # content similarity
            0.2 * hybrid_df['Weight'] / hybrid_df['Weight'].max()  # normalized popularity
        )
        
        # Sort by final score
        hybrid_df = hybrid_df.sort_values('final_score', ascending=False)
        
        # Print top recommendations
        for n,row in enumerate(hybrid_df.head(no_of_recommendation).itertuples(),1):
            print(f"{n}. {row.title}")
            print(f"Final Score : {round(row.final_score,4)}\n")

In [39]:
final_hybrid_recommendation("Toy Story (1995)",5)

Your selected movie : Toy Story (1995)
------------------------------------------------------------
1. Toy Story 2 (1999)
Final Score : 0.8511

2. Bug's Life, A (1998)
Final Score : 0.8217

3. Aladdin (1992)
Final Score : 0.7697

4. Lion King, The (1994)
Final Score : 0.7374

5. Beauty and the Beast (1991)
Final Score : 0.7344



## Evaluation of models:

In [40]:
#for evaluation model we want function "final_hybrid_recommendation" output as return not print so little changes done

def final_hybrid_recommendation(movie_name, no_of_recommendation=5):
    
    if movie_name not in pivot_table_rating.index:
        return (f"Your watched movie '{movie_name}' has total rating count either less than 50 or not in training data so not able to recommedate movie")
    else:
        #print ("Your selected movie :", movie_name)
        #print("-"*60)
        
        # ----- Step 1: Collaborative Filtering -----
        movie_index = pivot_table_rating.index.get_loc(movie_name)
        distances, indices = collab_model.kneighbors(
            pivot_table_rating.iloc[movie_index,:].values.reshape(1,-1),
            n_neighbors=20
        )
        movie_indices = indices[0][1:]  # ignore selected movie
        hybrid_df = pd.DataFrame({"title": pivot_table_rating.index[movie_indices]})
        
        # ----- Step 2: Merge TF-IDF content similarity score -----
        # Compute cosine similarity of selected movie with candidates

        # Create mapping from movie title to index in TF-IDF matrix
        indices_content = pd.Series(movie_df.index, index=movie_df['title']).to_dict()
        
        idx = indices_content.get(movie_name, None)
        if idx is not None:
            sim_scores = list(enumerate(cosine_sim[idx]))                                 #output[(0, np.float64(1.0000000000000002)),...]
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            sim_scores_dict = {movie_df['title'].iloc[i[0]]: i[1] for i in sim_scores}
            hybrid_df['content_sim'] = hybrid_df['title'].map(sim_scores_dict)            #map>>"For each title in hybrid_df['title'], find its value in sim_scores_dict and put it into content_sim column"
        else:
            hybrid_df['content_sim'] = 0
        
        # ----- Step 3: Merge popularity (Weighted Rating) -----
        hybrid_df = hybrid_df.merge(
            df[['title','Weight']].drop_duplicates(),
            on='title',
            how='left'
        )
        
        # ----- Step 4: Compute final score -----
        # Weighted sum: 50% collaborative, 30% content similarity, 20% popularity
        hybrid_df['final_score'] = (
            0.5 * (1 / (1 + distances[0][1:])) +  # collaborative (convert distance → similarity)
            0.3 * hybrid_df['content_sim'] +      # content similarity
            0.2 * hybrid_df['Weight'] / hybrid_df['Weight'].max()  # normalized popularity
        )
        
        # Sort by final score
        hybrid_df = hybrid_df.sort_values('final_score', ascending=False).reset_index()
        hybrid_df.index = hybrid_df.index + 1
        
        # Print top recommendations
        return hybrid_df[['title','final_score']].head(no_of_recommendation)

In [41]:
def evaluate_model():

    scores = []

    users = df['userId'].unique()[:20]        #taking 20 user
 
    for user in users:

        user_data = df[df['userId'] == user]

        # take highly rated movies
        liked_movies = user_data[user_data['rating'] >= 4]['title'].tolist()

        if len(liked_movies) < 2:
            continue

        test_movie = liked_movies[0]
        actual_movies = liked_movies[1:]

        try:
            recommended = final_hybrid_recommendation(test_movie, 20)['title'].tolist()
        except:
            continue

        # lower case for matching
        recommended = [i.lower() for i in recommended]
        actual_movies = [i.lower() for i in actual_movies]

        hit = len(set(recommended) & set(actual_movies)) / len(recommended)        #ratio of common in reco and like/all_recommendate

        scores.append(hit)
    print("Average Recommendation Score:", sum(scores)/len(scores))
    
evaluate_model()

Average Recommendation Score: 0.38421052631578945


## ML PIPELINE:

In [42]:
#run below code only if you want to create pipeline   #comment out this statement if want to run

import pandas as pd
import numpy as np
import joblib

from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
# -------------------------
# 1. Load Data
# -------------------------

def load_data():
    
    movies = pd.read_csv(
        "datasets/movies.dat",
        sep="::",
        encoding="latin-1",
        engine="python",
        names=['movieId', 'title', 'genres']
    )

    ratings = pd.read_csv(
        "datasets/ratings.dat",
        sep="::",
        engine="python",
        names=['userId', 'movieId', 'rating', 'timestamp']
    )

    users = pd.read_csv(
        "datasets/users.dat",
        sep="::",
        engine="python",
        names=['userId','gender','age','occupation','zip']
    )

    df = movies.merge(ratings, on="movieId")
    df = df.merge(users, on="userId")

    return df


# -------------------------
# 2. Data Preprocessing
# -------------------------

def preprocess(df):

    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

    df['title'] = df['title'].str.replace(
        r"^[\.\'\s ]+", "", regex=True
    ).str.strip()

    df['genres'] = df['genres'].str.replace('|',' ')
    df['genres'] = df['genres'].str.replace("Children's", "Childrens")
    df['genres'] = df['genres'].str.replace("Sci-Fi", "SciFi")
    df['genres'] = df['genres'].str.replace("Film-Noir", "FilmNoir")

    return df


# -------------------------
# 3. Feature Engineering
# -------------------------

def feature_engineering(df):

    counting = df.groupby('title')['rating'].count().reset_index()
    counting.rename(columns={'rating':'count'}, inplace=True)

    averaging = df.groupby('title')['rating'].mean().reset_index()
    averaging.rename(columns={'rating':'mean'}, inplace=True)

    df = df.merge(counting, on='title')
    df = df.merge(averaging, on='title')

    df = df[df['count'] > 50]

    C = df['mean'].mean()
    M = 50

    df['Weight'] = (
        (df['mean'] * df['count'] + C * M) /
        (df['count'] + M)
    )

    return df


# -------------------------
# 4. Collaborative Filtering
# -------------------------

def train_collaborative(df):

    pivot_table = pd.pivot_table(
        index='title',
        columns='userId',
        values='rating',
        data=df
    ).fillna(0)

    model = NearestNeighbors(
        metric='cosine',
        algorithm='brute',
        n_neighbors=20,
        n_jobs=-1
    )

    model.fit(pivot_table)

    return pivot_table, model


# -------------------------
# 5. Content Based Filtering
# -------------------------

def train_content(df):

    movie_df = df[['movieId','title','genres']].drop_duplicates()

    tfidf = TfidfVectorizer(stop_words='english')

    tfidf_matrix = tfidf.fit_transform(movie_df['genres'])

    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    return movie_df, tfidf, cosine_sim


# -------------------------
# 6. Save Models
# -------------------------

def save_models(
    pivot_table,
    collab_model,
    movie_df,
    tfidf,
    cosine_sim,
    df
):
    import os
    os.makedirs("models", exist_ok=True)
    joblib.dump(pivot_table, "models/pivot_table.pkl")
    joblib.dump(collab_model, "models/collab_model.pkl")
    joblib.dump(movie_df, "models/movie_df.pkl")
    joblib.dump(tfidf, "models/tfidf.pkl")
    joblib.dump(cosine_sim, "models/cosine_sim.pkl")
    joblib.dump(df, "models/final_df.pkl")


# -------------------------
# Main Pipeline
# -------------------------

def main():

    df = load_data()

    df = preprocess(df)

    df = feature_engineering(df)

    pivot_table, collab_model = train_collaborative(df)

    movie_df, tfidf, cosine_sim = train_content(df)

    save_models(
        pivot_table,
        collab_model,
        movie_df,
        tfidf,
        cosine_sim,
        df
    )


if __name__ == "__main__":
    main()

In [44]:
#Step 3 — Load Saved Models
import joblib
import pandas as pd
import numpy as np

# Load Models
pivot_table = joblib.load("models/pivot_table.pkl")
collab_model = joblib.load("models/collab_model.pkl")
movie_df = joblib.load("models/movie_df.pkl")
tfidf = joblib.load("models/tfidf.pkl")
cosine_sim = joblib.load("models/cosine_sim.pkl")
df = joblib.load("models/final_df.pkl")

In [45]:
def final_hybrid_recommendation(movie_name, no_of_recommendation=5):
    
    if movie_name not in pivot_table_rating.index:
        return (f"Your watched movie '{movie_name}' has total rating count either less than 50 or not in training data so not able to recommedate movie")
    else:
        #print ("Your selected movie :", movie_name)
        #print("-"*60)
        
        # ----- Step 1: Collaborative Filtering -----
        movie_index = pivot_table_rating.index.get_loc(movie_name)
        distances, indices = collab_model.kneighbors(
            pivot_table_rating.iloc[movie_index,:].values.reshape(1,-1),
            n_neighbors=20
        )
        movie_indices = indices[0][1:]  # ignore selected movie
        hybrid_df = pd.DataFrame({"title": pivot_table_rating.index[movie_indices]})
        
        # ----- Step 2: Merge TF-IDF content similarity score -----
        # Compute cosine similarity of selected movie with candidates

        # Create mapping from movie title to index in TF-IDF matrix
        indices_content = pd.Series(movie_df.index, index=movie_df['title']).to_dict()
        
        idx = indices_content.get(movie_name, None)
        if idx is not None:
            sim_scores = list(enumerate(cosine_sim[idx]))                                 #output[(0, np.float64(1.0000000000000002)),...]
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            sim_scores_dict = {movie_df['title'].iloc[i[0]]: i[1] for i in sim_scores}
            hybrid_df['content_sim'] = hybrid_df['title'].map(sim_scores_dict)            #map>>"For each title in hybrid_df['title'], find its value in sim_scores_dict and put it into content_sim column"
        else:
            hybrid_df['content_sim'] = 0
        
        # ----- Step 3: Merge popularity (Weighted Rating) -----
        hybrid_df = hybrid_df.merge(
            df[['title','Weight']].drop_duplicates(),
            on='title',
            how='left'
        )
        
        # ----- Step 4: Compute final score -----
        # Weighted sum: 50% collaborative, 30% content similarity, 20% popularity
        hybrid_df['final_score'] = (
            0.5 * (1 / (1 + distances[0][1:])) +  # collaborative (convert distance → similarity)
            0.3 * hybrid_df['content_sim'] +      # content similarity
            0.2 * hybrid_df['Weight'] / hybrid_df['Weight'].max()  # normalized popularity
        )
        
        # Sort by final score
        hybrid_df = hybrid_df.sort_values('final_score', ascending=False).reset_index()
        hybrid_df.index = hybrid_df.index + 1
        
        # Print top recommendations
        return hybrid_df[['title','final_score']].head(no_of_recommendation)

In [46]:
#Step 5 — Test Your Function
print(final_hybrid_recommendation("Toy Story (1995)",5))


Your selected movie : Toy Story (1995)
                         title  final_score
1           Toy Story 2 (1999)     0.851073
2         Bug's Life, A (1998)     0.821673
3               Aladdin (1992)     0.769656
4        Lion King, The (1994)     0.737366
5  Beauty and the Beast (1991)     0.734377


What This Evaluation Is Doing (Simple Idea)

We are checking:

Pick a user

Find movies they liked (rating ≥ 4)

Take one movie they liked

Ask your model to recommend similar movies

Check if recommended movies are also liked by that user

Repeat for many users

Take average score

That’s it. That's how we got 0.4

In [47]:
def evaluate_model():

    scores = []

    users = df['userId'].unique()[:20]        #taking 20 user
 
    for user in users:

        user_data = df[df['userId'] == user]

        # take highly rated movies
        liked_movies = user_data[user_data['rating'] >= 4]['title'].tolist()

        if len(liked_movies) < 2:
            continue

        test_movie = liked_movies[0]
        actual_movies = liked_movies[1:]

        try:
            recommended = final_hybrid_recommendation(test_movie, 5)['title'].tolist()
        except:
            continue

        # lower case for matching
        recommended = [i.lower() for i in recommended]
        actual_movies = [i.lower() for i in actual_movies]

        hit = len(set(recommended) & set(actual_movies)) / len(recommended)        #ratio of common in reco and like/all_recommendate

        scores.append(hit)
    print("Average Recommendation Score:", sum(scores)/len(scores))


evaluate_model()

Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Star Wars: Episode VI - Return of the Jedi (1983)
Your selected movie : Toy Story (1995)
Your selected movie : Father of the Bride Part II (1995)
Your selected movie : Clueless (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Your selected movie : Toy Story (1995)
Average Recommendation Score: 0.49411764705882355


## __Create ml module function for fastapi__


In [16]:
import joblib
import os 
# create models directory if not exists
os.makedirs("../models", exist_ok=True)
joblib.dump(pivot_table_rating, "../models/pivot_table_rating.pkl")
joblib.dump(collab_model, "../models/collab_model.pkl")
joblib.dump(movie_df, "../models/movie_df.pkl")
joblib.dump(cosine_sim, "../models/cosine_sim.pkl")
joblib.dump(df, "../models/df.pkl")

['../models/df.pkl']

In [17]:
pivot_table_rating = joblib.load("../models/pivot_table_rating.pkl")
collab_model = joblib.load("../models/collab_model.pkl")
movie_df = joblib.load("../models/movie_df.pkl")
cosine_sim = joblib.load("../models/cosine_sim.pkl")
df = joblib.load("../models/df.pkl")

In [18]:
import pandas as pd

#loadin_save_file

class MovieRecommender:

    def __init__(self):
        self.pivot_table_rating = pivot_table_rating
        self.collab_model = collab_model
        self.movie_df = movie_df
        self.cosine_sim = cosine_sim
        self.df = df

    # ------------------------------
    # Hybrid Recommendation Function
    # ------------------------------
    def final_hybrid_recommendation(self, movie_name, no_of_recommendation=5):

        if movie_name not in self.pivot_table_rating.index:
            return f"Movie '{movie_name}' not in training data"

        # Step 1: Collaborative Filtering
        movie_index = self.pivot_table_rating.index.get_loc(movie_name)

        distances, indices = self.collab_model.kneighbors(
            self.pivot_table_rating.iloc[movie_index, :].values.reshape(1, -1),
            n_neighbors=20
        )

        movie_indices = indices[0][1:]

        hybrid_df = pd.DataFrame({
            "title": self.pivot_table_rating.index[movie_indices]
        })

        # Step 2: Content Similarity
        indices_content = pd.Series(
            self.movie_df.index,
            index=self.movie_df['title']
        ).to_dict()

        idx = indices_content.get(movie_name, None)

        if idx is not None:
            sim_scores = list(enumerate(self.cosine_sim[idx]))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

            sim_scores_dict = {
                self.movie_df['title'].iloc[i[0]]: i[1]
                for i in sim_scores
            }

            hybrid_df['content_sim'] = hybrid_df['title'].map(sim_scores_dict)

        else:
            hybrid_df['content_sim'] = 0

        # Step 3: Popularity
        hybrid_df = hybrid_df.merge(
            self.df[['title', 'Weight']].drop_duplicates(),
            on='title',
            how='left'
        )

        # Step 4: Final Score
        hybrid_df['final_score'] = (
            0.5 * (1 / (1 + distances[0][1:])) +
            0.3 * hybrid_df['content_sim'] +
            0.2 * hybrid_df['Weight'] / hybrid_df['Weight'].max()
        )

        hybrid_df = hybrid_df.sort_values(
            'final_score',
            ascending=False
        ).reset_index()

        return hybrid_df[['title', 'final_score']].head(no_of_recommendation)


    # ------------------------------
    # Model Evaluation
    # ------------------------------
    def evaluate_model(self):

        scores = []

        users = self.df['userId'].unique()[:20]

        for user in users:

            user_data = self.df[self.df['userId'] == user]

            liked_movies = user_data[
                user_data['rating'] >= 4
            ]['title'].tolist()

            if len(liked_movies) < 2:
                continue

            test_movie = liked_movies[0]
            actual_movies = liked_movies[1:]

            try:
                recommended = self.final_hybrid_recommendation(
                    test_movie,
                    20
                )['title'].tolist()

            except:
                continue

            recommended = [i.lower() for i in recommended]
            actual_movies = [i.lower() for i in actual_movies]

            hit = len(set(recommended) & set(actual_movies)) / len(recommended)

            scores.append(hit)

        return sum(scores) / len(scores)

In [19]:
# load all your models/data here
model = MovieRecommender()

In [20]:
model.final_hybrid_recommendation('Toy Story (1995)',5)

,title,final_score
0,Toy Story 2 (1999),0.851073
1,"Bug's Life, A (1998)",0.821673
2,Aladdin (1992),0.769656
3,"Lion King, The (1994)",0.737366
4,Beauty and the Beast (1991),0.734377
